# Covariance

DTU provided covariance information for the CHAOS-8.6 model. 

This can be used to generate realistic perturbations for the synthetic record.

Here a set of perturbations are generated for SV gauss-time measurements, over the full resolved record

# Preamble

In [2]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
# setting up file path for imports
import sys
import os
from pathlib import Path

cwd = Path(os.getcwd())
print(cwd)
PROJECT_ROOT = cwd.resolve().parents[2]
sys.path.insert(0, str(PROJECT_ROOT))

# installing relevant libraries
import chaosmagpy as cp
import numpy as np
import matplotlib.pyplot as plt
import h5py
from pydmd import DMD, BOPDMD
from pydmd.preprocessing import hankel_preprocessing

# Project paths / utilities
from src.msc_thesis.paths import *
from src.msc_thesis.synSetup import *
from src.msc_thesis.synUtils import *
from src.msc_thesis.synDMD import *

/home/elliott/projects/msc_thesis_project/Last_stretch/Data_prep/Non_signal_Data
['/home/elliott/projects/msc_thesis_project', '/home/elliott/miniforge3/envs/msc_thesis/lib/python311.zip', '/home/elliott/miniforge3/envs/msc_thesis/lib/python3.11', '/home/elliott/miniforge3/envs/msc_thesis/lib/python3.11/lib-dynload', '', '/home/elliott/miniforge3/envs/msc_thesis/lib/python3.11/site-packages', '/home/elliott/projects/msc_thesis_project']
['/home/elliott/projects/msc_thesis_project', '/home/elliott/projects/msc_thesis_project', '/home/elliott/miniforge3/envs/msc_thesis/lib/python311.zip', '/home/elliott/miniforge3/envs/msc_thesis/lib/python3.11', '/home/elliott/miniforge3/envs/msc_thesis/lib/python3.11/lib-dynload', '', '/home/elliott/miniforge3/envs/msc_thesis/lib/python3.11/site-packages', '/home/elliott/projects/msc_thesis_project']
There are 70 knots
There are: 64 splines
There are: 20 degrees
['/home/elliott/projects/msc_thesis_project', '/home/elliott/projects/msc_thesis_project', 

In [4]:
# reading in the (smaller) covariance file information provided

file_path = Path(f"{CHAOS_COV_DIR}/CHAOS_Cov_1997_2026_0806_BSpl.h5")

with h5py.File(file_path, "r") as cov_file:

    knots = np.asarray(cov_file["knots"])
    n_spl =  np.asarray(cov_file["n_spl"])
    n_m =  np.asarray(cov_file["n_m"])
    nmax =  np.asarray(cov_file["nmax"])
    order =  np.asarray(cov_file["order"])


    print(f"There are {len(knots)} knots")
    print(f"There are: {n_spl} splines")
    print(f"There are: {nmax} degrees")
        

There are 70 knots
There are: 64 splines
There are: 20 degrees


In [5]:
# defining used parameters:

# collocation matrix formation
H = cp.model_utils.colloc_matrix(x=times_used_mjd, knots=knots, order=order, deriv = 1)
print(np.allclose(H, H_sv))

H_sv_T = H_sv.T

False


In [6]:
print(np.shape(H))

(53, 64)


In [7]:
print(times_used)

[2000.  2000.5 2001.  2001.5 2002.  2002.5 2003.  2003.5 2004.  2004.5
 2005.  2005.5 2006.  2006.5 2007.  2007.5 2008.  2008.5 2009.  2009.5
 2010.  2010.5 2011.  2011.5 2012.  2012.5 2013.  2013.5 2014.  2014.5
 2015.  2015.5 2016.  2016.5 2017.  2017.5 2018.  2018.5 2019.  2019.5
 2020.  2020.5 2021.  2021.5 2022.  2022.5 2023.  2023.5 2024.  2024.5
 2025.  2025.5 2026. ]


In [8]:
# defining used parameters:

file_path = Path(f"{CHAOS_COV_DIR}/CHAOS_Cov_1997_2026_0806_BSpl.h5")

# collocation matrix formation
H = cp.model_utils.colloc_matrix(x=times_used_mjd, knots=knots, order=order, deriv = 1)
print(np.allclose(H, H_sv)) # same as H_sv calculated elsewhere

with h5py.File(file_path, "r") as cov_file:
    C = cov_file["C"]

False


In [9]:
from pathlib import Path

import h5py
import numpy as np


# ---------------------------------------------------------------------
# Paths and truncation parameters
# ---------------------------------------------------------------------

source_path = (
    Path(CHAOS_COV_DIR)
    / "CHAOS_Cov_1997_2026_0806_BSpl.h5"
)

output_path = (
    Path(CHAOS_COV_DIR)
    / "CHAOS_Cov_1997_2026_0806_BSpl_nmax15_marginal.h5"
)

nmax_keep = 15
block_rows = 128  # ~16 MiB per block for float64 and 16,320 columns


def copy_attributes(source, destination):
    """Copy all HDF5 attributes from one object to another."""
    for key, value in source.attrs.items():
        destination.attrs[key] = value


def read_scalar(dataset):
    """Read an HDF5 scalar dataset as a Python scalar."""
    value = dataset[()]
    return value.item() if isinstance(value, np.generic) else value


def create_replacement_scalar(
    source_file,
    output_file,
    name,
    value,
):
    """
    Create an updated scalar dataset while retaining its original dtype
    and attributes where possible.
    """
    if name in output_file:
        del output_file[name]

    if name in source_file:
        source_dataset = source_file[name]
        dtype = source_dataset.dtype
        output_dataset = output_file.create_dataset(
            name,
            data=np.asarray(value, dtype=dtype),
        )
        copy_attributes(source_dataset, output_dataset)
    else:
        output_file.create_dataset(name, data=value)


# ---------------------------------------------------------------------
# Extract the degree <= 15 marginal covariance
# ---------------------------------------------------------------------

with h5py.File(source_path, "r") as source_file:

    # Read n_spl from the file rather than hard-coding it.
    if "n_spl" not in source_file:
        raise KeyError("The source file does not contain an 'n_spl' dataset.")

    n_spl = int(read_scalar(source_file["n_spl"]))
    n_gauss_keep = nmax_keep * (nmax_keep + 2)
    n_model_keep = n_gauss_keep * n_spl

    source_C = source_file["C"]

    if source_C.ndim != 2 or source_C.shape[0] != source_C.shape[1]:
        raise ValueError(
            f"C must be square, but has shape {source_C.shape}."
        )

    if n_model_keep > source_C.shape[0]:
        raise ValueError(
            f"Requested {n_model_keep} model parameters, but C only "
            f"contains {source_C.shape[0]}."
        )

    print(f"Source covariance shape: {source_C.shape}")
    print(f"Keeping nmax:            {nmax_keep}")
    print(f"Gauss coefficients:      {n_gauss_keep}")
    print(f"Splines per coefficient: {n_spl}")
    print(
        f"Marginal covariance:     "
        f"({n_model_keep}, {n_model_keep})"
    )

    with h5py.File(output_path, "w") as output_file:

        # Copy root-level attributes.
        copy_attributes(source_file, output_file)

        # Record what the reduced file contains.
        output_file.attrs["description"] = (
            "Marginal CHAOS B-spline covariance for spherical-harmonic "
            f"degrees n <= {nmax_keep}."
        )
        output_file.attrs["source_file"] = source_path.name
        output_file.attrs["marginal_nmax"] = nmax_keep
        output_file.attrs["n_gauss_coefficients"] = n_gauss_keep

        # Copy unchanged datasets/groups first.
        # C, m, nmax and n_m are recreated below.
        replaced_names = {
            "C",
            "m",
            "nmax",
            "n_m",
            "n_gnm",
        }

        for name in source_file:
            if name not in replaced_names:
                source_file.copy(
                    source_file[name],
                    output_file,
                    name=name,
                )

        # -------------------------------------------------------------
        # Create the reduced covariance dataset
        # -------------------------------------------------------------

        chunk_rows = min(block_rows, n_model_keep)
        chunk_cols = min(1024, n_model_keep)

        output_C = output_file.create_dataset(
            "C",
            shape=(n_model_keep, n_model_keep),
            dtype=source_C.dtype,
            chunks=(chunk_rows, chunk_cols),
            # Optional fast lossless compression:
            # compression="lzf",
            shuffle=True,
        )
        copy_attributes(source_C, output_C)

        output_C.attrs["description"] = (
            f"Marginal covariance for Gauss coefficients through "
            f"degree {nmax_keep}, arranged as {n_gauss_keep} blocks "
            f"of {n_spl} B-spline coefficients."
        )

        # Copy the leading principal submatrix one row block at a time.
        for row_start in range(0, n_model_keep, block_rows):
            row_stop = min(row_start + block_rows, n_model_keep)

            output_C[row_start:row_stop, :] = source_C[
                row_start:row_stop,
                :n_model_keep,
            ]

            print(
                f"\rCopied rows {row_stop:,}/{n_model_keep:,}",
                end="",
                flush=True,
            )

        print()

        # -------------------------------------------------------------
        # Truncate the mean/model vector consistently
        # -------------------------------------------------------------

        if "m" in source_file:
            source_m = source_file["m"]

            if source_m.shape[0] < n_model_keep:
                raise ValueError(
                    f"m has length {source_m.shape[0]}, but "
                    f"{n_model_keep} entries are required."
                )

            output_m = output_file.create_dataset(
                "m",
                data=source_m[:n_model_keep],
                dtype=source_m.dtype,
            )
            copy_attributes(source_m, output_m)

            output_m.attrs["description"] = (
                f"B-spline coefficients through spherical-harmonic "
                f"degree {nmax_keep}."
            )

        # Update metadata describing the reduced model.
        create_replacement_scalar(
            source_file,
            output_file,
            "nmax",
            nmax_keep,
        )

        create_replacement_scalar(
            source_file,
            output_file,
            "n_m",
            n_model_keep,
        )

        # Create/update this only if it is present in the original file.
        if "n_gnm" in source_file:
            create_replacement_scalar(
                source_file,
                output_file,
                "n_gnm",
                n_gauss_keep,
            )

        output_file.flush()


print(f"Saved reduced covariance to:\n{output_path}")

Source covariance shape: (28160, 28160)
Keeping nmax:            15
Gauss coefficients:      255
Splines per coefficient: 64
Marginal covariance:     (16320, 16320)
Copied rows 16,320/16,320
Saved reduced covariance to:
/home/elliott/projects/msc_thesis_project/data/chaos_covariance/CHAOS_Cov_1997_2026_0806_BSpl_nmax15_marginal.h5


In [10]:
with h5py.File(output_path, "r") as reduced_file:
    print("Datasets:", list(reduced_file.keys()))
    print("C shape:", reduced_file["C"].shape)
    print("m shape:", reduced_file["m"].shape)
    print("nmax:", reduced_file["nmax"][()])
    print("n_m:", reduced_file["n_m"][()])

    assert reduced_file["C"].shape == (16_320, 16_320)
    assert reduced_file["m"].shape == (16_320,)
    assert int(reduced_file["nmax"][()]) == 15
    assert int(reduced_file["n_m"][()]) == 16_320

Datasets: ['C', 'knots', 'm', 'n_m', 'n_spl', 'nmax', 'order', 'version']
C shape: (16320, 16320)
m shape: (16320,)
nmax: 15
n_m: 16320


In [11]:
with (
    h5py.File(source_path, "r") as source_file,
    h5py.File(output_path, "r") as reduced_file,
):
    test_source = source_file["C"][5000:5100, 9000:9100]
    test_reduced = reduced_file["C"][5000:5100, 9000:9100]

    assert np.array_equal(test_source, test_reduced)

print("Small-block verification passed.")

Small-block verification passed.


In [12]:
from scipy.linalg import cholesky

Ng = 15 * (15 + 2)  # 255
Ns = 64
Nt = len(times_used_mjd)

H = cp.model_utils.colloc_matrix(
    x=times_used_mjd,
    knots=knots,
    order=order,
    deriv=1,
)

assert H.shape == (Nt, Ns)
assert Nt == 53

with h5py.File(output_path, "r") as reduced_file:
    C15 = reduced_file["C"][:]
    m15 = reduced_file["m"][:]

C15 = 0.5 * (C15 + C15.T)

rng = np.random.default_rng(1234)

L15 = cholesky(C15, lower=True)
delta_beta = (L15 @ rng.standard_normal(Ng * Ns)).reshape(Ng, Ns)

# Shape: (255 Gauss coefficients, 53 epochs)
delta_gauss_time = delta_beta @ H.T

In [13]:
delta_gauss_time_grid = A_15_r @ delta_gauss_time

Cube_Movie(delta_gauss_time_grid, name='perturbations')

'/home/elliott/projects/msc_thesis_project/figures/perturbations.mp4'

In [14]:
from pathlib import Path

import h5py
import numpy as np


# Relevant output filename
L15_path = (
    Path(CHAOS_COV_DIR)
    / "CHAOS_Cov_1997_2026_0806_BSpl_nmax15_Cholesky_lower.h5"
)

Ng = 15 * (15 + 2)   # 255
Ns = 64
n_model = Ng * Ns    # 16,320

assert L15.shape == (n_model, n_model)
assert np.issubdtype(L15.dtype, np.floating)

block_rows = 128

with h5py.File(L15_path, "w") as f:
    L_dataset = f.create_dataset(
        "L",
        shape=L15.shape,
        dtype=L15.dtype,
        chunks=(block_rows, min(1024, n_model)),
        compression="lzf",  # fast, lossless; remove for maximum write speed
        shuffle=True,
    )

    # Write in row blocks to avoid creating another large temporary array
    for row_start in range(0, n_model, block_rows):
        row_stop = min(row_start + block_rows, n_model)

        L_dataset[row_start:row_stop, :] = L15[row_start:row_stop, :]

        print(
            f"\rWritten rows {row_stop:,}/{n_model:,}",
            end="",
            flush=True,
        )

    print()

    # File metadata
    f.attrs["description"] = (
        "Lower-triangular Cholesky factor of the marginal CHAOS "
        "B-spline covariance through spherical-harmonic degree 15."
    )
    f.attrs["source_covariance_file"] = (
        "CHAOS_Cov_1997_2026_0806_BSpl_nmax15_marginal.h5"
    )
    f.attrs["factorisation"] = "C15 = L @ L.T"
    f.attrs["triangle"] = "lower"
    f.attrs["nmax"] = 15
    f.attrs["n_gauss_coefficients"] = Ng
    f.attrs["n_spl"] = Ns
    f.attrs["n_model_parameters"] = n_model

    L_dataset.attrs["description"] = (
        "Lower-triangular Cholesky factor L, satisfying C15 = L @ L.T."
    )

print(f"Saved Cholesky factor to:\n{L15_path}")

Written rows 16,320/16,320
Saved Cholesky factor to:
/home/elliott/projects/msc_thesis_project/data/chaos_covariance/CHAOS_Cov_1997_2026_0806_BSpl_nmax15_Cholesky_lower.h5


In [15]:
# example of loading it in later

rng = np.random.default_rng(1234)
z = rng.standard_normal(n_model)

delta_beta_flat = np.empty(n_model, dtype=L15.dtype)

with h5py.File(L15_path, "r") as f:
    L = f["L"]

    for row_start in range(0, n_model, block_rows):
        row_stop = min(row_start + block_rows, n_model)

        delta_beta_flat[row_start:row_stop] = (
            L[row_start:row_stop, :] @ z
        )

H = (
    365.25
    * cp.model_utils.colloc_matrix(
        x=times_used_mjd,
        knots=knots,
        order=order,
        deriv=1,
    )
)


delta_beta = delta_beta_flat.reshape(Ng, Ns)
delta_gauss_time = delta_beta @ H.T  # shape (255, 53)

In [16]:
Cube_Movie(A_15_r @ delta_gauss_time)

'/home/elliott/projects/msc_thesis_project/figures/data_cube.mp4'

In [17]:
from pathlib import Path

import chaosmagpy as cp
import h5py
import numpy as np


# ---------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------

n_real = 1000
nmax = 15

Ng = nmax * (nmax + 2)   # 255 Gauss coefficients
Ns = 64                   # B-spline coefficients per Gauss coefficient

seed = 12345
batch_size = 32
derivative_order = 1

times_used_mjd = np.asarray(times_used_mjd, dtype=np.float64)
knots = np.asarray(knots, dtype=np.float64)
order = int(order)

Nt = times_used_mjd.size
n_model = Ng * Ns         # 16,320

output_path = (
    Path(CHAOS_COV_DIR)
    / (
        f"CHAOS_0806_nmax{nmax}_SV_nTyr_"
        f"perturbations_N{n_real}_Nt{Nt}.h5"
    )
)

output_path.parent.mkdir(parents=True, exist_ok=True)


# ---------------------------------------------------------------------
# Validate the Cholesky factor
# ---------------------------------------------------------------------

if L15.shape != (n_model, n_model):
    raise ValueError(
        f"Expected L15 shape {(n_model, n_model)}, "
        f"but received {L15.shape}."
    )

if L15.dtype not in (np.dtype("float32"), np.dtype("float64")):
    raise TypeError(
        f"L15 must contain float32 or float64 values, not {L15.dtype}."
    )

if not np.all(np.isfinite(L15)):
    raise ValueError("L15 contains NaN or infinite values.")

# The calculation assumes:
#
#     C15 = L15 @ L15.T
#
# i.e. L15 is the lower-triangular Cholesky factor.


# ---------------------------------------------------------------------
# Validate the temporal sampling
# ---------------------------------------------------------------------

if times_used_mjd.ndim != 1:
    raise ValueError(
        f"times_used_mjd must be one-dimensional, "
        f"but has shape {times_used_mjd.shape}."
    )

if Nt != 53:
    raise ValueError(f"Expected Nt=53, but received Nt={Nt}.")

if not np.all(np.isfinite(times_used_mjd)):
    raise ValueError("times_used_mjd contains NaN or infinite values.")

if np.any(np.diff(times_used_mjd) <= 0):
    raise ValueError("times_used_mjd must be strictly increasing.")

# For a B-spline of order k, the valid model interval is bounded by
# knots[k - 1] and knots[-k].
valid_time_start = knots[order - 1]
valid_time_end = knots[-order]

if times_used_mjd.min() < valid_time_start:
    raise ValueError(
        f"First requested epoch {times_used_mjd.min()} is before the "
        f"spline interval start {valid_time_start}."
    )

if times_used_mjd.max() > valid_time_end:
    raise ValueError(
        f"Last requested epoch {times_used_mjd.max()} is after the "
        f"spline interval end {valid_time_end}."
    )


# ---------------------------------------------------------------------
# Construct the SV collocation matrix
# ---------------------------------------------------------------------

# colloc_matrix(..., deriv=1) differentiates with respect to the
# MJD2000 coordinate in days. Multiplication by 365.25 converts
# the derivative basis from per day to per year.
H_sv = (
    365.25
    * cp.model_utils.colloc_matrix(
        x=times_used_mjd,
        knots=knots,
        order=order,
        deriv=derivative_order,
    )
)

if H_sv.shape != (Nt, Ns):
    raise ValueError(
        f"Expected H_sv shape {(Nt, Ns)}, "
        f"but received {H_sv.shape}."
    )

if not np.all(np.isfinite(H_sv)):
    raise ValueError(
        "H_sv contains NaN or infinite values. Check that all requested "
        "epochs lie within the spline model interval."
    )


# ---------------------------------------------------------------------
# Calculation dtype
# ---------------------------------------------------------------------

# Retain the dtype of L15 to avoid creating a second, potentially
# multi-gigabyte copy of the Cholesky factor.
calculation_dtype = L15.dtype
L15_used = L15

# H_sv is small, so converting it is inexpensive.
H_used = np.asarray(H_sv, dtype=calculation_dtype)

rng = np.random.default_rng(seed)


# ---------------------------------------------------------------------
# Avoid accidental overwrite
# ---------------------------------------------------------------------

if output_path.exists():
    raise FileExistsError(
        f"Output file already exists:\n{output_path}\n"
        "Delete it explicitly or change the filename."
    )


# ---------------------------------------------------------------------
# Generate and save perturbations
# ---------------------------------------------------------------------

with h5py.File(output_path, "w") as f:

    perturbation_dataset = f.create_dataset(
        "perturbations",
        shape=(n_real, Ng, Nt),
        dtype=calculation_dtype,

        # Optimised for retrieving individual realisations.
        chunks=(1, Ng, Nt),

        # Gaussian values generally do not compress substantially.
        compression=None,

        # Per-chunk checksum.
        fletcher32=True,
    )

    time_dataset = f.create_dataset(
        "times_mjd2000",
        data=times_used_mjd,
    )

    H_dataset = f.create_dataset(
        "H_sv",
        data=H_used,
    )

    knots_dataset = f.create_dataset(
        "knots_mjd2000",
        data=knots,
    )

    f.create_dataset(
        "order",
        data=order,
    )

    # -----------------------------------------------------------------
    # File metadata
    # -----------------------------------------------------------------

    f.attrs["description"] = (
        "Zero-mean Gaussian CHAOS-8.6 secular-variation perturbation "
        "realisations through spherical-harmonic degree 15."
    )

    f.attrs["source_model"] = "CHAOS-8.6 / 0806"

    f.attrs["source_factor_file"] = (
        "CHAOS_Cov_1997_2026_0806_BSpl_"
        "nmax15_Cholesky_lower.h5"
    )

    f.attrs["factorisation_convention"] = (
        "C15 = L15 @ L15.T"
    )

    f.attrs["n_real"] = n_real
    f.attrs["nmax"] = nmax
    f.attrs["n_gauss_coefficients"] = Ng
    f.attrs["n_spl"] = Ns
    f.attrs["n_times"] = Nt
    f.attrs["derivative_order"] = derivative_order
    f.attrs["derivative_time_unit"] = "year"
    f.attrs["mjd2000_days_per_year"] = 365.25
    f.attrs["random_seed"] = seed
    f.attrs["batch_size"] = batch_size

    f.attrs["array_ordering"] = (
        "perturbations[realisation, gauss_coefficient, time]"
    )

    f.attrs["gauss_coefficient_ordering"] = (
        "g10, g11, h11, g20, g21, h21, ..."
    )

    perturbation_dataset.attrs["description"] = (
        "Zero-mean Gaussian perturbations of secular-variation "
        "Gauss coefficients."
    )
    perturbation_dataset.attrs["units"] = "nT/year"
    perturbation_dataset.attrs["shape_order"] = (
        "realisation, gauss coefficient, time"
    )

    time_dataset.attrs["units"] = "days"
    time_dataset.attrs["epoch"] = "2000-01-01 00:00:00 UT"
    time_dataset.attrs["description"] = "MJD2000 evaluation epochs"

    H_dataset.attrs["description"] = (
        "First-derivative B-spline collocation matrix scaled by "
        "365.25 to give derivatives per year."
    )
    H_dataset.attrs["units"] = "1/year"
    H_dataset.attrs["derivative_order"] = derivative_order

    knots_dataset.attrs["units"] = "MJD2000 days"

    # -----------------------------------------------------------------
    # Generation loop
    # -----------------------------------------------------------------

    for start in range(0, n_real, batch_size):

        stop = min(start + batch_size, n_real)
        current_batch = stop - start

        # Each column is an independent N(0, I) vector.
        #
        # Shape:
        #     (16,320 spline parameters, current batch size)
        Z = rng.standard_normal(
            size=(n_model, current_batch),
            dtype=calculation_dtype,
        )

        # Correlated B-spline coefficient perturbations:
        #
        #     delta_beta = L15 z
        #
        # Shape:
        #     (16,320, current batch size)
        delta_beta_flat = L15_used @ Z

        # Convert to:
        #
        #     (batch, Gauss coefficient, spline coefficient)
        #
        # Ordering remains:
        #     g10 splines, g11 splines, h11 splines, ...
        delta_beta = (
            delta_beta_flat.T
            .reshape(current_batch, Ng, Ns)
        )

        # Project each spline expansion onto the requested epochs:
        #
        #     (batch, Ng, Ns) @ (Ns, Nt)
        #         -> (batch, Ng, Nt)
        #
        # Units: nT/year
        delta_sv = delta_beta @ H_used.T

        perturbation_dataset[start:stop, :, :] = delta_sv

        print(
            f"\rGenerated and saved {stop:,}/{n_real:,} realisations",
            end="",
            flush=True,
        )

    print()
    f.flush()


size_mib = output_path.stat().st_size / 1024**2

print(f"Saved perturbations to:\n{output_path}")
print(f"File size: {size_mib:.2f} MiB")

Generated and saved 1,000/1,000 realisations
Saved perturbations to:
/home/elliott/projects/msc_thesis_project/data/chaos_covariance/CHAOS_0806_nmax15_SV_nTyr_perturbations_N1000_Nt53.h5
File size: 103.21 MiB


In [18]:
print(np.allclose((365.25 * cp.model_utils.colloc_matrix(x=times_res_mjd,knots=knots,order=order,deriv=1,)), R_Read_In("H", "SV")))

True
